In [1]:
print('eh')

eh


In [2]:
import os 
from langchain_core.prompts import ChatMessagePromptTemplate,ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import OllamaLLM
from langchain_core.runnables import RunnablePassthrough,RunnableBranch

In [3]:
def get_llm():
    return OllamaLLM(
        model="tinyllama:1.1b",
        temperature=0
    )

llm=get_llm()

In [4]:
# --- Define Simulated Sub-Agent Handlers (equivalent to ADK sub_agents) --- 

def booking_handler(request: str) -> str: 
   """Simulates the Booking Agent handling a request.""" 
   print("\n--- DELEGATING TO BOOKING HANDLER ---") 
   return f"Booking Handler processed request: '{request}'. Result: Simulated booking action." 

In [5]:
def info_handler(request: str) -> str: 
   """Simulates the Info Agent handling a request.""" 
   print("\n--- DELEGATING TO INFO HANDLER ---") 
   return f"Info Handler processed request: '{request}'. Result: Simulated information retrieval."  

In [6]:
def unclear_handler(request: str) -> str: 
   """Handles requests that couldn't be delegated.""" 
   print("\n--- HANDLING UNCLEAR REQUEST ---") 
   return f"Coordinator could not delegate request: '{request}'. Please clarify." 

In [7]:
# --- Define Coordinator Router Chain (equivalent to ADK coordinator's instruction) --- 
# This chain decides which handler to delegate to. 
coordinator_router_prompt = ChatPromptTemplate.from_messages([ 
   ("system", """Analyze the user's request and determine which specialist handler should process it. 
    - If the request is related to booking flights or hotels,  
      output 'booker'. 
    - For all other general information questions, output 'info'. 
    - If the request is unclear or doesn't fit either category,  
      output 'unclear'. 
    ONLY output one word: 'booker', 'info', or 'unclear'."""), 
   ("user", "{request}") 
]) 

In [8]:
if llm:
    coordinator_router_chain=coordinator_router_prompt|llm|StrOutputParser()

In [9]:
# DEfine the Delegation Logic 
#Define RunnableBranch to route based on the router chains output.

#define the branches for the RunnableBranch

branches={
    'booker':RunnablePassthrough.assign(output=lambda x:booking_handler(x['request']['request'])),
    'info':RunnablePassthrough.assign(output=lambda x:info_handler(x['request']['request'])),
    'unclear':RunnablePassthrough.assign(output=lambda x:unclear_handler(x['request']['request'])),    
}


In [10]:
#create the runnable branch. it takes the output of the router chain
#and routes the original input ('request') to the corresponding handler.

delegation_branch=RunnableBranch(
    (lambda x: x['decision'].strip()=='booker',branches['booker']),
    (lambda x: x['decision'].strip()=='info',branches['info']),
    branches['unclear'] #default branches for 'unclear' or any other output
)

In [11]:
#combine the router chain and the delegation branch into a single runnable
#the router chain's outputa('decision') is passed along with the original input ('request')
#to the delegation_branch

coordinator_agent={
    'decision':coordinator_router_chain,
    'request':RunnablePassthrough()
} | delegation_branch | (lambda x: x['output'])  #extract the final output





In [12]:
#Example usage

def main():
    print('running with a booking request')
    request_a ='Book me a hotel  in  london'
    result_a=coordinator_agent.invoke({'request':request_a})
    print(f'final result a: {result_a}')
    
    print('running with an info request')
    request_b='what is the capital city of nepal'
    result_b=coordinator_agent.invoke({'request':request_b})
    print(f'final result b: {result_b}')
    
    
    print('running with an unclear request')
    request_c="Tell me about quantum physics"
    result_c=coordinator_agent.invoke({'request':request_c})
    print(f'final result c:{result_c}')   




In [ ]:
if __name__=='__main__':
    main()

running with a booking request

--- HANDLING UNCLEAR REQUEST ---
final result a: Coordinator could not delegate request: 'Book me a hotel  in  london'. Please clarify.
running with an info request

--- HANDLING UNCLEAR REQUEST ---
final result b: Coordinator could not delegate request: 'what is the capital city of nepal'. Please clarify.
running with an unclear request
